# Music Annotation Project V7.9.1: Notebook 22a NPZ Metric Recovery and Merge, V2

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook recovers the completed Notebook 22 Stage 4 run **without rerunning reconstruction**.

It:

1. locates the existing V7.9.1 run containing `finalists/` and `stage4_pages/`;
2. verifies the complete V7.5 cache roster and authoritative arrays required for NPZ validation;
3. identifies missing, zero-byte, unparseable or schema-invalid page CSVs;
4. validates every required NPZ by opening it and checking keys, shapes, route partition and outside-mask immutability;
5. recalculates page and component metrics from the saved reconstruction/routing arrays;
6. writes non-empty, fixed-schema CSVs atomically;
7. replaces each repaired completion marker with semantic hashes and row counts;
8. merges all page outputs and creates the final challenger summary and diagnostic queue.

The notebook never calls reconstruction or inpainting functions. It does not read frozen V7.8 independent data and does not modify Notebook 09.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy


## Configuration and source-run discovery


In [ ]:
import cv2, json, hashlib, gc, shutil
import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V79_1_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9_1'
SOURCE_RUN=None  # Set to the exact completed/crashed V7.9.1 run if automatic selection is wrong.
FORCE_REGENERATE_ALL_METRICS=False
MIN_COMPONENT_AREA=20

PROHIBITED=[
 'independent_structural_validation_v7_8_builder',
 'independent_structural_validation_v7_8_rebuilt',
 'independent_policy_validation_v7_8',
]
PAGE_COLUMNS=['page','candidate','mae','ink_precision','ink_recall','ink_f1','edge_precision','edge_recall','edge_f1','true_print_preserved','false_annotation_preserved','preserve_precision','glyph_pixels','removed_clean_print_fraction','outside_mask_drift']
COMPONENT_COLUMNS=['page','component','candidate','x','y','w','h','area','ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved']
REQUIRED_NPZ_KEYS={'reconstruction','preserve','remove','ambiguous','glyph'}

def assert_development_path(path):
    s=str(Path(path)).lower()
    if any(x in s for x in PROHIBITED):raise RuntimeError(f'Prohibited V7.8 independent path: {path}')

def latest_run(root,required):
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

if SOURCE_RUN is None:
    SOURCE_RUN=latest_run(V79_1_ROOT,['finalists','stage4_pages','tables','all_page_confirmation_route_metrics_checkpoint.csv'])
if SOURCE_RUN is None:
    # Some runs keep the confirmation checkpoint in tables only.
    SOURCE_RUN=latest_run(V79_1_ROOT,['finalists','stage4_pages','tables/all_page_confirmation_route_metrics_checkpoint.csv'])
if SOURCE_RUN is None:raise RuntimeError('Set SOURCE_RUN to the V7.9.1 run containing finalists/ and stage4_pages/.')
SOURCE_RUN=Path(SOURCE_RUN);assert_development_path(SOURCE_RUN)
FINALISTS=SOURCE_RUN/'finalists';PAGE_RESULTS=SOURCE_RUN/'stage4_pages';TABLES=SOURCE_RUN/'tables';RECOVERY=SOURCE_RUN/'recovery_22a';BACKUP=RECOVERY/'replaced_files'
for p in [RECOVERY,BACKUP]:p.mkdir(parents=True,exist_ok=True)
print('SOURCE_RUN:',SOURCE_RUN)
print('FINALISTS:',FINALISTS)
print('PAGE_RESULTS:',PAGE_RESULTS)
print('TABLES:',TABLES)


## Helpers and authoritative development inputs


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):
    assert_development_path(folder)
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
    assert_development_path(path)
    with np.load(path,allow_pickle=False) as z:return {k:z[k] for k in z.files}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def atomic_csv(df,path,columns):
    frame=df.reindex(columns=columns);tmp=Path(str(path)+'.tmp');frame.to_csv(tmp,index=False);tmp.replace(path)
def atomic_json(value,path):
    tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,default=str));tmp.replace(path)
def backup(path):
    if not path.exists():return None
    target=BACKUP/path.name
    if target.exists():target=BACKUP/f'{path.stem}_{datetime.now().strftime("%H%M%S%f")}{path.suffix}'
    shutil.copy2(path,target);return target

def latest_v75():return latest_run(V75_ROOT,['tables/cache_status.csv'])
V75=latest_v75()
if V75 is None:raise RuntimeError('No complete V7.5 development cache found.')
assert_development_path(V75)
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean');observed_map=image_map(ROOT/'aligned')

roles_path=TABLES/'development_page_roles.csv'
if not roles_path.exists():
    # Fall back to pages represented by Stage 4 NPZs.
    pages=sorted({p.name.split('_v78_baseline.npz')[0] for p in FINALISTS.glob('*_v78_baseline.npz')})
else:
    pages=pd.read_csv(roles_path).page.astype(str).tolist()
if not pages:raise RuntimeError('No development pages found.')

manifest_path=SOURCE_RUN/'run_manifest.json'
if manifest_path.exists():
    manifest=json.loads(manifest_path.read_text());selected_candidates=list(manifest.get('selected_candidates',[]))
else:selected_candidates=[]
if not selected_candidates:
    confirmation_path=TABLES/'all_page_confirmation_route_metrics_checkpoint.csv'
    if confirmation_path.exists():
        cf=pd.read_csv(confirmation_path);selected_candidates=sorted(x for x in cf.candidate.astype(str).unique() if x!='v78_baseline' and all((FINALISTS/f'{p}_{x}.npz').exists() for p in pages))
if len(selected_candidates)!=3:
    # Derive candidates present for every page.
    common=None
    for page in pages:
        names={p.name[len(page)+1:-4] for p in FINALISTS.glob(f'{page}_*.npz') if p.name.endswith('.npz')}
        common=names if common is None else common&names
    selected_candidates=sorted(x for x in (common or set()) if x!='v78_baseline')
if len(selected_candidates)!=3:raise RuntimeError(f'Expected three challengers, found {selected_candidates}')
CANDIDATES=['v78_baseline']+selected_candidates
print('V7.5 reference:',V75)
print('Pages:',len(pages))
print('Candidates:',CANDIDATES)


## Verify V7.5 cache provenance and classify broken CSVs


In [ ]:
# Upstream provenance gate: complete V7.5 cache plus direct semantic
# validation of every saved reconstruction/routing NPZ.
required_v75_columns={'page','cache'}
missing_status_columns=required_v75_columns-set(status.columns)
if missing_status_columns:raise RuntimeError(f'V7.5 cache_status.csv missing columns: {sorted(missing_status_columns)}')
if status.page.astype(str).duplicated().any():raise RuntimeError('V7.5 cache_status.csv contains duplicate page values.')
v75_provenance=[]
for page in pages:
    row={'page':page,'cache_listed':page in cache_lookup,'cache_exists':False,'required_arrays_present':False,'shape_compatible':False,'error':''}
    try:
        if page not in cache_lookup:raise RuntimeError('page absent from V7.5 cache_status.csv')
        cache_path=cache_lookup[page];row['cache_exists']=cache_path.exists()
        if not cache_path.exists():raise RuntimeError(f'V7.5 cache missing: {cache_path}')
        arrays=loadz(cache_path);required={'mask','telea','geometry_t80'};missing=sorted(required-set(arrays));row['required_arrays_present']=not missing
        if missing:raise RuntimeError('missing V7.5 arrays: '+','.join(missing))
        if page not in clean_map or page not in observed_map:raise RuntimeError('clean or observed development page missing')
        clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0)
        if clean is None or observed is None:raise RuntimeError('clean or observed development page unreadable')
        row['shape_compatible']=bool(clean.shape==observed.shape==arrays['mask'].shape==arrays['telea'].shape==arrays['geometry_t80'].shape)
        if not row['shape_compatible']:raise RuntimeError('V7.5 cache/page shape mismatch')
        row['telea_outside_mask_immutable']=bool(np.array_equal(arrays['telea'][arrays['mask']==0],observed[arrays['mask']==0]))
        row['geometry_outside_mask_immutable']=bool(np.array_equal(arrays['geometry_t80'][arrays['mask']==0],observed[arrays['mask']==0]))
        if not row['telea_outside_mask_immutable'] or not row['geometry_outside_mask_immutable']:raise RuntimeError('V7.5 cache violates outside-mask immutability')
        row['valid']=True
    except Exception as e:row['valid']=False;row['error']=str(e)
    v75_provenance.append(row)
v75_provenance_df=pd.DataFrame(v75_provenance);v75_provenance_df.to_csv(RECOVERY/'v7_5_cache_provenance.csv',index=False)
invalid_v75=v75_provenance_df[~v75_provenance_df.valid.astype(bool)]
if not invalid_v75.empty:
    display(invalid_v75);raise RuntimeError(f'V7.5 cache provenance failed for {len(invalid_v75)} page(s).')
print('V7.5 cache provenance PASSED:',len(v75_provenance_df),'pages')

def csv_state(path,required_columns,expected_rows=None):
    if not path.exists():return 'missing',None
    if path.stat().st_size==0:return 'zero_byte',None
    try:df=pd.read_csv(path)
    except EmptyDataError:return 'empty_data',None
    except Exception as e:return f'unparseable:{type(e).__name__}',None
    missing=[x for x in required_columns if x not in df.columns]
    if missing:return 'schema_missing:'+','.join(missing),df
    if expected_rows is not None and len(df)!=expected_rows:return f'row_count:{len(df)}',df
    return 'valid',df
classification=[]
for page in pages:
    ms,_=csv_state(PAGE_RESULTS/f'{page}_metrics.csv',PAGE_COLUMNS,len(CANDIDATES));cs,_=csv_state(PAGE_RESULTS/f'{page}_components.csv',COMPONENT_COLUMNS,None)
    classification.append({'page':page,'metrics_state':ms,'components_state':cs,'needs_recovery':FORCE_REGENERATE_ALL_METRICS or ms!='valid' or cs!='valid'})
classification_df=pd.DataFrame(classification);classification_df.to_csv(RECOVERY/'pre_recovery_csv_classification.csv',index=False)
print(classification_df.metrics_state.value_counts(dropna=False));print(classification_df.components_state.value_counts(dropna=False));print('Pages requiring metric regeneration:',int(classification_df.needs_recovery.sum()));display(classification_df[classification_df.needs_recovery])


## Validate saved NPZ reconstruction and routing artefacts

Validation is semantic rather than based on ZIP container size alone. Required arrays must open, match the authoritative page shape, form an exact preserve/remove/ambiguous partition of the mask, contain no routed pixels outside the mask, and leave reconstruction pixels outside the mask byte-for-byte unchanged.


In [ ]:
npz_audit=[]
for page in pages:
    if page not in clean_map or page not in observed_map or page not in cache_lookup:raise RuntimeError(f'{page}: missing controlled development inputs')
    observed=cv2.imread(str(observed_map[page]),0);zref=loadz(cache_lookup[page]);mask=zref['mask'];shape=observed.shape
    for candidate in CANDIDATES:
        path=FINALISTS/f'{page}_{candidate}.npz';row={'page':page,'candidate':candidate,'path':str(path),'exists':path.exists(),'bytes':path.stat().st_size if path.exists() else 0}
        try:
            arrays=loadz(path);row['keys']='|'.join(sorted(arrays));missing=sorted(REQUIRED_NPZ_KEYS-set(arrays));row['missing_keys']='|'.join(missing)
            if missing:raise RuntimeError('missing keys '+','.join(missing))
            recon=arrays['reconstruction'];preserve=arrays['preserve']>0;remove=arrays['remove']>0;ambiguous=arrays['ambiguous']>0;glyph=arrays['glyph']>0;m=mask>0
            row.update({'shape_ok':all(arrays[k].shape==shape for k in REQUIRED_NPZ_KEYS),'partition_exact':bool(np.array_equal((preserve.astype(np.uint8)+remove.astype(np.uint8)+ambiguous.astype(np.uint8)),m.astype(np.uint8))),'routes_outside_mask':int(np.sum((preserve|remove|ambiguous|glyph)&~m)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m])),'reconstruction_sha256':array_hash(recon),'preserve_sha256':array_hash(preserve.astype(np.uint8)),'remove_sha256':array_hash(remove.astype(np.uint8)),'ambiguous_sha256':array_hash(ambiguous.astype(np.uint8)),'glyph_sha256':array_hash(glyph.astype(np.uint8)),'valid':True,'error':''})
            if not row['shape_ok'] or not row['partition_exact'] or row['routes_outside_mask'] or row['outside_mask_drift']:raise RuntimeError('semantic validation failed')
        except Exception as e:
            row['valid']=False;row['error']=str(e)
        npz_audit.append(row)
    del observed,zref,mask;gc.collect()
npz_audit_df=pd.DataFrame(npz_audit);npz_audit_df.to_csv(RECOVERY/'npz_semantic_validation.csv',index=False)
failed=npz_audit_df[~npz_audit_df.valid.astype(bool)]
if not failed.empty:
    display(failed);raise RuntimeError(f'NPZ semantic validation failed for {len(failed)} page/candidate files. No CSVs were replaced.')
print('NPZ semantic validation PASSED:',len(npz_audit_df),'files')


## Regenerate only missing or invalid page metrics from NPZ arrays


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_scores(ref,pred,region,tol=1):
    region=region.astype(bool);ref=ref&region;pred=pred&region
    if not region.any() or not ref.any():return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(int(pred.sum()),1));recall=float(np.sum(ref&pd)/int(ref.sum()));return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))
def page_metrics(page,candidate,clean,observed,mask,arrays,clean_ink,clean_edges):
    recon=arrays['reconstruction'];preserve=arrays['preserve']>0;remove=arrays['remove']>0;glyph=arrays['glyph']>0;m=mask>0
    recon_ink=printed_ink(recon);recon_edges=cv2.Canny(recon,70,170)>0;ip,ir,if1=tolerant_scores(clean_ink,recon_ink,m);ep,er,ef1=tolerant_scores(clean_edges,recon_edges,m);diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));tp=int(np.sum(preserve&clean_ink));fp=int(np.sum(preserve&~clean_ink))
    return {'page':page,'candidate':candidate,'mae':float(diff[m].mean()) if m.any() else np.nan,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'true_print_preserved':tp,'false_annotation_preserved':fp,'preserve_precision':tp/max(tp+fp,1),'glyph_pixels':int(glyph.sum()),'removed_clean_print_fraction':float(np.sum(remove&clean_ink&m)/max(int(m.sum()),1)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m]))}

def component_metrics(page,candidate,clean,mask,arrays,labels,stats,clean_ink,clean_edges):
    recon=arrays['reconstruction'];preserve=arrays['preserve']>0;recon_ink=printed_ink(recon);recon_edges=cv2.Canny(recon,70,170)>0;diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));rows=[]
    for cid in range(1,len(stats)):
        x,y,w,h,a=[int(v) for v in stats[cid]]
        if a<MIN_COMPONENT_AREA:continue
        comp=labels[y:y+h,x:x+w]==cid
        if not comp.any():continue
        ip,ir,if1=tolerant_scores(clean_ink[y:y+h,x:x+w],recon_ink[y:y+h,x:x+w],comp);ep,er,ef1=tolerant_scores(clean_edges[y:y+h,x:x+w],recon_edges[y:y+h,x:x+w],comp);values=diff[y:y+h,x:x+w][comp]
        rows.append({'page':page,'component':cid,'candidate':candidate,'x':x,'y':y,'w':w,'h':h,'area':a,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'mae':float(values.mean()) if values.size else np.nan,'false_annotation_preserved':int(np.sum(preserve[y:y+h,x:x+w]&comp&~clean_ink[y:y+h,x:x+w]))})
    return rows

repair_rows=[]
for _,item in classification_df.iterrows():
    page=str(item.page)
    if not bool(item.needs_recovery):
        repair_rows.append({'page':page,'action':'retained_valid_csvs','page_rows':len(CANDIDATES),'component_rows':None});continue
    clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);zref=loadz(cache_lookup[page]);mask=zref['mask'];clean_ink=printed_ink(clean);clean_edges=cv2.Canny(clean,70,170)>0;n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    page_rows=[];component_rows=[]
    for candidate in CANDIDATES:
        arrays=loadz(FINALISTS/f'{page}_{candidate}.npz');page_rows.append(page_metrics(page,candidate,clean,observed,mask,arrays,clean_ink,clean_edges));component_rows.extend(component_metrics(page,candidate,clean,mask,arrays,labels,stats,clean_ink,clean_edges));del arrays;gc.collect()
    metrics_path=PAGE_RESULTS/f'{page}_metrics.csv';components_path=PAGE_RESULTS/f'{page}_components.csv';marker_path=PAGE_RESULTS/f'{page}_complete.json'
    backups=[str(x) for x in [backup(metrics_path),backup(components_path),backup(marker_path)] if x is not None]
    atomic_csv(pd.DataFrame(page_rows),metrics_path,PAGE_COLUMNS);atomic_csv(pd.DataFrame(component_rows),components_path,COMPONENT_COLUMNS)
    # Re-open and validate before writing the completion marker.
    mdf=pd.read_csv(metrics_path);cdf=pd.read_csv(components_path)
    if len(mdf)!=len(CANDIDATES) or set(mdf.candidate)!=set(CANDIDATES):raise RuntimeError(f'{page}: regenerated page metrics validation failed')
    if list(cdf.columns)!=COMPONENT_COLUMNS:raise RuntimeError(f'{page}: regenerated component schema validation failed')
    outputs={str(metrics_path.relative_to(SOURCE_RUN)):sha256_file(metrics_path),str(components_path.relative_to(SOURCE_RUN)):sha256_file(components_path)}
    for candidate in CANDIDATES:
        p=FINALISTS/f'{page}_{candidate}.npz';outputs[str(p.relative_to(SOURCE_RUN))]=sha256_file(p)
    marker={'page':page,'candidates':selected_candidates,'recovered_by':'22a_V7_9_1_NPZ_Metric_Recovery_and_Merge','completed_utc':datetime.now(timezone.utc).isoformat(),'page_metric_rows':len(mdf),'component_metric_rows':len(cdf),'outputs':outputs}
    atomic_json(marker,marker_path)
    repair_rows.append({'page':page,'action':'regenerated_metrics_from_npz','page_rows':len(mdf),'component_rows':len(cdf),'backups':'|'.join(backups)})
    print('Recovered',page,'page rows',len(mdf),'component rows',len(cdf));del clean,observed,zref,mask,clean_ink,clean_edges,labels,stats,page_rows,component_rows,mdf,cdf;gc.collect()
repair_df=pd.DataFrame(repair_rows);repair_df.to_csv(RECOVERY/'metric_recovery_log.csv',index=False)
print(repair_df.action.value_counts())


## Final validation, merge and completion of Notebook 22


In [ ]:
page_frames=[];component_frames=[];post=[]
for page in pages:
    mp=PAGE_RESULTS/f'{page}_metrics.csv';cp=PAGE_RESULTS/f'{page}_components.csv';ms,mdf=csv_state(mp,PAGE_COLUMNS,len(CANDIDATES));cs,cdf=csv_state(cp,COMPONENT_COLUMNS,None)
    valid=(ms=='valid' and cs=='valid' and set(mdf.candidate)==set(CANDIDATES))
    post.append({'page':page,'metrics_state':ms,'components_state':cs,'candidate_set_ok':bool(ms=='valid' and set(mdf.candidate)==set(CANDIDATES)),'valid':valid})
    if not valid:continue
    page_frames.append(mdf);component_frames.append(cdf)
post_df=pd.DataFrame(post);post_df.to_csv(RECOVERY/'post_recovery_validation.csv',index=False)
if not post_df.valid.all():display(post_df[~post_df.valid]);raise RuntimeError('Recovery incomplete; merged tables were not written.')
page_df=pd.concat(page_frames,ignore_index=True);component_df=pd.concat(component_frames,ignore_index=True) if component_frames else pd.DataFrame(columns=COMPONENT_COLUMNS)
expected_page_rows=len(pages)*len(CANDIDATES)
if len(page_df)!=expected_page_rows:raise RuntimeError(f'Expected {expected_page_rows} page rows, found {len(page_df)}')
if page_df.duplicated(['page','candidate']).any():raise RuntimeError('Duplicate page/candidate rows found.')
if int(page_df.outside_mask_drift.fillna(0).max())!=0:raise RuntimeError('Outside-mask drift found in merged page metrics.')
atomic_csv(page_df,TABLES/'stage4_page_metrics.csv',PAGE_COLUMNS);atomic_csv(component_df,TABLES/'stage4_component_metrics.csv',COMPONENT_COLUMNS)

final=page_df.groupby('candidate',as_index=False).agg(pages=('page','nunique'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),mae=('mae','mean'),false_annotation_preserved=('false_annotation_preserved','sum'),true_print_preserved=('true_print_preserved','sum'),preserve_precision=('preserve_precision','mean'),outside_mask_drift=('outside_mask_drift','max'))
base=final[final.candidate=='v78_baseline']
if len(base)!=1:raise RuntimeError('Expected exactly one frozen V7.8 summary row.')
base=base.iloc[0]
for metric in ['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved']:
    final['delta_'+metric]=final[metric]-base[metric]
final.to_csv(TABLES/'stage4_finalist_summary.csv',index=False)

queue=[]
challengers=page_df[page_df.candidate!='v78_baseline']
for candidate,g in challengers.groupby('candidate'):
    for label,column,ascending in [('largest_false_preservation','false_annotation_preserved',False),('largest_true_print_preserved','true_print_preserved',False),('lowest_ink_recall','ink_recall',True)]:
        ranked=g.dropna(subset=[column]).sort_values(column,ascending=ascending).head(12)
        for rank,(_,row) in enumerate(ranked.iterrows(),1):queue.append({'category':label,'rank':rank,'candidate':candidate,'page':row.page,'value':row[column]})
pd.DataFrame(queue).to_csv(TABLES/'diagnostic_review_queue.csv',index=False)

manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'22a-v7.9.1-npz-metric-recovery-v2','source_run':str(SOURCE_RUN),'v7_5_reference':str(V75),'pages':len(pages),'candidates':CANDIDATES,'v7_5_cache_pages_validated':len(v75_provenance_df),'baseline_summary_csv_required':False,'npz_files_validated':len(npz_audit_df),'pages_recovered':int((repair_df.action=='regenerated_metrics_from_npz').sum()),'reconstruction_rerun':False,'outside_mask_drift_max':int(page_df.outside_mask_drift.fillna(0).max()),'merged_page_rows':len(page_df),'merged_component_rows':len(component_df),'notebook_09_modified':False,'v7_8_independent_data_read':False,'promotion_authorised':False,'outputs':{'page_metrics':str(TABLES/'stage4_page_metrics.csv'),'component_metrics':str(TABLES/'stage4_component_metrics.csv'),'finalist_summary':str(TABLES/'stage4_finalist_summary.csv'),'diagnostic_queue':str(TABLES/'diagnostic_review_queue.csv'),'npz_audit':str(RECOVERY/'npz_semantic_validation.csv'),'recovery_log':str(RECOVERY/'metric_recovery_log.csv'),'v7_5_cache_provenance':str(RECOVERY/'v7_5_cache_provenance.csv')}}
atomic_json(manifest,RECOVERY/'recovery_manifest.json')
print('RECOVERY AND MERGE COMPLETE')
print('Source run:',SOURCE_RUN)
print('Recovered pages:',manifest['pages_recovered'])
print('Merged page rows:',len(page_df))
print('Merged component rows:',len(component_df))
print('Final summary:',TABLES/'stage4_finalist_summary.csv')
display(final)
